In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType, DoubleType

In [4]:
# Создание SparkSession
spark = SparkSession.builder.appName("BikeAnalysis").getOrCreate()
spark

## Загрузка данных

In [5]:
# Загружаем данные
trips = spark.read.option("header", True).option("inferSchema", True).csv("trips.csv")
stations = spark.read.option("header", True).option("inferSchema", True).csv("stations.csv")

# Преобразуем типы
trips = trips.withColumn("duration", F.col("duration").cast("int"))
stations = stations.withColumn("lat", F.col("lat").cast("double"))
stations = stations.withColumn("long", F.col("long").cast("double"))

## Найти велосипед с максимальным временем пробега.

In [16]:
# Группируем по id велосипеда, суммируем время, сортируем по убыванию
top_bike = trips.groupBy("bike_id") \
  .agg(F.sum("duration").alias("total_duration")) \
  .orderBy(F.desc("total_duration")) \

top_bike_id = top_bike.first()['bike_id']

top_bike.show(1)

+-------+--------------+
|bike_id|total_duration|
+-------+--------------+
|    535|      18611693|
+-------+--------------+
only showing top 1 row


## Найти наибольшее геодезическое расстояние между станциями.

In [25]:
st1 = stations.alias("st1")
st2 = stations.alias("st2")

# Считаем по формуле гаверсинуса
# d = 2 * R * arcsin( sqrt( sin^2(dLat/2) + cos(lat1) * cos(lat2) * sin^2(dLon/2) ) )
# 6371 - средний радиус Земли
distances = st1.crossJoin(st2) \
    .filter(F.col("st1.id") < F.col("st2.id")) \
    .withColumn("lat1", F.radians(F.col("st1.lat"))) \
    .withColumn("lat2", F.radians(F.col("st2.lat"))) \
    .withColumn("dLat", F.radians(F.col("st2.lat") - F.col("st1.lat"))) \
    .withColumn("dLon", F.radians(F.col("st2.long") - F.col("st1.long"))) \
    .withColumn(
        "distance_km",
        2 * 6371 * F.asin(
            F.sqrt(
                F.pow(F.sin(F.col("dLat") / 2), 2) +
                F.cos(F.col("lat1")) * F.cos(F.col("lat2")) *
                F.pow(F.sin(F.col("dLon") / 2), 2)
            )
        )
    ) \
    .select(
        F.col("st1.name").alias("station1_name"),
        F.col("st2.name").alias("station2_name"),
        F.col("distance_km")
    )

max_distance = distances.orderBy(F.desc("distance_km"))
max_distance.show(1)

+--------------------+--------------------+----------------+
|       station1_name|       station2_name|     distance_km|
+--------------------+--------------------+----------------+
|SJSU - San Salvad...|Embarcadero at Sa...|69.9208759542826|
+--------------------+--------------------+----------------+
only showing top 1 row


## Найти путь велосипеда с максимальным временем пробега через станции.

In [30]:
# Фильтруем поездки по найденному bike_id, сортируем по дате
path = trips.filter(F.col("bike_id") == top_bike_id) \
    .orderBy("start_date") \
    .select("start_date", "start_station_name", "end_station_name")

# Отображаем первые 5 для читаемости
path.show(5, truncate=False)

3. Путь велосипеда-рекордсмена (первые 5 поездок):
+---------------+-----------------------------------+----------------------------------------+
|start_date     |start_station_name                 |end_station_name                        |
+---------------+-----------------------------------+----------------------------------------+
|1/1/2014 13:42 |Mechanics Plaza (Market at Battery)|Embarcadero at Sansome                  |
|1/1/2014 18:51 |Embarcadero at Sansome             |Market at 4th                           |
|1/1/2014 19:48 |Market at 4th                      |South Van Ness at Market                |
|1/10/2014 20:13|Market at 10th                     |Powell Street BART                      |
|1/10/2014 8:09 |Embarcadero at Folsom              |San Francisco Caltrain (Townsend at 4th)|
+---------------+-----------------------------------+----------------------------------------+
only showing top 5 rows


## Найти количество велосипедов в системе.

In [31]:
# Считаем уникальные bike_id
bike_count = trips.select("bike_id").distinct().count()
bike_count

700

## Найти пользователей потративших на поездки более 3 часов.

In [33]:
# 3 часа = 10800 секунд
users_over_3_hours = trips.filter(F.col("zip_code").isNotNull()) \
    .groupBy("zip_code") \
    .agg(F.sum("duration").alias("total_duration")) \
    .filter(F.col("total_duration") > 10800)

users_over_3_hours.show()

+--------+--------------+
|zip_code|total_duration|
+--------+--------------+
|   94102|      19128021|
|   95134|        728023|
|   84606|         95145|
|   80305|        180906|
|   60070|         28919|
|   95519|         30303|
|   43085|         11670|
|   91910|         50488|
|   77339|         13713|
|   48063|         13755|
|   85022|         12682|
|    1090|         20391|
|    2136|         16010|
|   11722|         24331|
|   95138|        155295|
|   94610|       3630628|
|   94404|       3589350|
|   80301|        152189|
|   91326|         65885|
|   90742|         10965|
+--------+--------------+
only showing top 20 rows
